# 12 — ML Filter Portfolio Backtest

Model sınıflandırma performansı zayıf fakat tamamen sıfır değildir. Bu nedenle
modeli kabul veya reddetme kararı sınıflandırma tablosundan değil, final Robot
portföy backtestinden verilecektir.

Bu notebook:

1. Development döneminde eğitilmiş geçici champion modeli yükler.
2. Validation dönemindeki bütün günlük Robot AL satırlarını skorlar.
3. Ham olasılık eşiklerini ve günlük göreli sıralama filtrelerini test eder.
4. ML'nin kabul edilmesi için katı portföy kriterleri uygular.
5. Validation'da seçilen tek konfigürasyonu Development+Validation ile yeniden
   eğitilmiş model üzerinden Audit döneminde raporlar.
6. ML fayda sağlamazsa orijinal Robot'u final sistem olarak korur.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import load

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src").is_dir()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

from src.features import add_indicators
from src.signals import (
    build_market_regime,
    add_robot_scores,
)
from src.presets import (
    FINAL_STRATEGY_CONFIG,
    FINAL_PORTFOLIO_CONFIG,
)
from src.ml_dataset import (
    BASE_FEATURE_COLUMNS,
    add_meta_features,
)
from src.ml_training import (
    build_candidate_models,
    train_on_development_predict_validation,
)
from src.ml_portfolio import (
    MLFilterConfig,
    add_model_probabilities,
    run_filter_grid,
    add_baseline_differences,
    validation_acceptance_table,
    select_validation_champion,
)
from src.benchmark import build_benchmark_equity
from src.metrics import portfolio_metrics


## 1. Fiyatları, meta-label olaylarını ve geçici modeli yükle


In [ ]:
stock_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "bist100_robot_clean.parquet"
)

market_prices = pd.read_parquet(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "xu100_robot_clean.parquet"
)

events = pd.read_parquet(
    PROJECT_ROOT
    / "results"
    / "ml"
    / "robot_meta_label_training.parquet"
)

for column in [
    "Signal_Date",
    "Entry_Date",
    "Exit_Date",
]:
    events[column] = pd.to_datetime(events[column])

MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "meta_label_candidate_model.joblib"
)

METADATA_PATH = (
    PROJECT_ROOT
    / "models"
    / "meta_label_candidate_metadata.json"
)

development_model = load(MODEL_PATH)

with METADATA_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    model_metadata = json.load(file)

champion_model_name = model_metadata["model_name"]

print("Geçici champion:", champion_model_name)
print("Model eğitim sonu:", model_metadata["training_end"])


## 2. Bütün günlük Robot AL satırları için ML özelliklerini oluştur


In [ ]:
stock_features = add_indicators(stock_prices)
market_features = add_indicators(market_prices)
market_regime = build_market_regime(market_features)

scored_prices = add_robot_scores(
    stock_features=stock_features,
    market_regime=market_regime,
    config=FINAL_STRATEGY_CONFIG,
    include_reasons=False,
)

featured_prices = add_meta_features(
    scored_prices=scored_prices,
    market_features=market_features,
)

print("Günlük özellikli satır:", len(featured_prices))
print("Toplam Robot AL satırı:", featured_prices["Signal"].eq("AL").sum())


## 3. Validation dönemini Development modeliyle skorla


In [ ]:
VALIDATION_START = "2023-01-01"
VALIDATION_END = "2024-12-31"

validation_prices = add_model_probabilities(
    featured_prices=featured_prices,
    fitted_model=development_model,
    feature_columns=BASE_FEATURE_COLUMNS,
    start=VALIDATION_START,
    end=VALIDATION_END,
)

validation_probability_summary = (
    validation_prices.loc[
        validation_prices["Date"].between(
            VALIDATION_START,
            VALIDATION_END,
        )
        & validation_prices["Signal"].eq("AL"),
        "ML_Probability",
    ]
    .describe()
)

display(validation_probability_summary)


## 4. Validation filtre grid'i

Ham olasılıkların kalibrasyonu zayıf olduğu için iki yaklaşım test edilir:

- Sabit olasılık eşiği
- Her işlem gününde en yüksek olasılıklı adayların belirli bir kısmı

`Baseline_Robot` hiçbir ML filtresi uygulamaz.


In [ ]:
filter_grid = [
    MLFilterConfig(
        name="Baseline_Robot",
    ),
    MLFilterConfig(
        name="Threshold_0.400",
        probability_threshold=0.400,
    ),
    MLFilterConfig(
        name="Threshold_0.425",
        probability_threshold=0.425,
    ),
    MLFilterConfig(
        name="Threshold_0.450",
        probability_threshold=0.450,
    ),
    MLFilterConfig(
        name="Threshold_0.475",
        probability_threshold=0.475,
    ),
    MLFilterConfig(
        name="Threshold_0.500",
        probability_threshold=0.500,
    ),
    MLFilterConfig(
        name="Daily_Top_75pct",
        keep_top_fraction=0.75,
    ),
    MLFilterConfig(
        name="Daily_Top_50pct",
        keep_top_fraction=0.50,
    ),
    MLFilterConfig(
        name="Daily_Top_30pct",
        keep_top_fraction=0.30,
    ),
]

validation_results, validation_outputs = (
    run_filter_grid(
        probability_prices=validation_prices,
        filters=filter_grid,
        strategy_config=FINAL_STRATEGY_CONFIG,
        portfolio_config=FINAL_PORTFOLIO_CONFIG,
        start=VALIDATION_START,
        end=VALIDATION_END,
    )
)

validation_results = add_baseline_differences(
    validation_results
)

validation_columns = [
    "Filter_Name",
    "CAGR_%",
    "Max_Drawdown_%",
    "Profit_Factor",
    "Sharpe",
    "Calmar",
    "Trade_Count",
    "Signal_Pass_Rate_%",
    "CAGR_%_Difference",
    "Max_Drawdown_%_Difference",
    "Profit_Factor_Difference",
    "Calmar_Difference",
]

display(
    validation_results[
        validation_columns
    ].sort_values(
        ["Calmar", "CAGR_%"],
        ascending=False,
    )
)


## 5. Katı ML kabul kriterleri


In [ ]:
acceptance_table = validation_acceptance_table(
    validation_results=validation_results,
    baseline_name="Baseline_Robot",
    minimum_trade_fraction=0.50,
    maximum_drawdown_deterioration_pp=3.0,
)

display(
    acceptance_table[
        [
            "Filter_Name",
            "CAGR_%",
            "Max_Drawdown_%",
            "Profit_Factor",
            "Sharpe",
            "Calmar",
            "Trade_Count",
            "Signal_Pass_Rate_%",
            "Enough_Trades",
            "CAGR_Improved",
            "Calmar_Improved",
            "Profit_Factor_Not_Worse",
            "Drawdown_Acceptable",
            "Acceptance_Count",
            "ML_Accepted",
        ]
    ]
)

selected_filter_name, ml_accepted = (
    select_validation_champion(
        acceptance_table
    )
)

print("Validation kararı:", selected_filter_name)
print("ML kabul edildi mi?:", ml_accepted)


ML filtresi şu şartların tamamını sağlamalıdır:

1. Baseline işlem sayısının en az %50'sini korumalı.
2. Validation CAGR yükselmeli.
3. Validation Calmar yükselmeli.
4. Profit Factor baseline'dan düşük olmamalı.
5. Drawdown en fazla 3 puan kötüleşmeli.

Hiçbir filtre bunları sağlamazsa final sistem orijinal Robot olarak kalır.


## 6. Validation champion eğri karşılaştırması


In [ ]:
baseline_validation_equity = (
    validation_outputs["Baseline_Robot"]["equity"]
)

selected_validation_equity = (
    validation_outputs[selected_filter_name]["equity"]
)

validation_chart = (
    baseline_validation_equity[
        ["Date", "Equity"]
    ]
    .rename(
        columns={
            "Equity": "Baseline Robot"
        }
    )
    .merge(
        selected_validation_equity[
            ["Date", "Equity"]
        ].rename(
            columns={
                "Equity": selected_filter_name
            }
        ),
        on="Date",
        how="inner",
    )
    .set_index("Date")
)

plt.figure(figsize=(12, 6))
plt.plot(
    validation_chart.index,
    validation_chart["Baseline Robot"],
    label="Baseline Robot",
)
plt.plot(
    validation_chart.index,
    validation_chart[selected_filter_name],
    label=selected_filter_name,
)
plt.title("Validation — Robot ve ML Filtresi")
plt.xlabel("Tarih")
plt.ylabel("Portföy Değeri (TL)")
plt.legend()
plt.tight_layout()
plt.show()


## 7. Development + Validation ile yeniden eğit ve Audit'i skorla

Audit sonucu model veya eşik seçmek için kullanılmaz. Validation'da verilen
kararın ileri dönem davranışını raporlamak için kullanılır.


In [ ]:
AUDIT_START = "2025-01-01"
AUDIT_END = min(
    featured_prices["Date"].max(),
    market_prices["Date"].max(),
).strftime("%Y-%m-%d")

development_validation_events = (
    events.loc[
        events["Period"].isin(
            ["Development", "Validation"]
        )
    ]
    .sort_values(
        ["Signal_Date", "Ticker"]
    )
    .reset_index(drop=True)
)

audit_events = (
    events.loc[
        events["Period"].eq(
            "Audit_2025_Plus"
        )
    ]
    .sort_values(
        ["Signal_Date", "Ticker"]
    )
    .reset_index(drop=True)
)

all_candidate_models = build_candidate_models(
    feature_columns=BASE_FEATURE_COLUMNS,
    random_state=42,
)

selected_model_dictionary = {
    champion_model_name: all_candidate_models[
        champion_model_name
    ]
}

(
    audit_fitted_models,
    audit_event_metrics,
    audit_event_predictions,
) = train_on_development_predict_validation(
    development_data=development_validation_events,
    validation_data=audit_events,
    feature_columns=BASE_FEATURE_COLUMNS,
    target_column="Meta_Label",
    models=selected_model_dictionary,
    cutoff_date=AUDIT_START,
    embargo_days=5,
)

audit_model = audit_fitted_models[
    champion_model_name
]

display(audit_event_metrics)


In [ ]:
audit_prices = add_model_probabilities(
    featured_prices=featured_prices,
    fitted_model=audit_model,
    feature_columns=BASE_FEATURE_COLUMNS,
    start=AUDIT_START,
    end=AUDIT_END,
)

selected_filter_config = next(
    filter_config
    for filter_config in filter_grid
    if filter_config.name == selected_filter_name
)

audit_filters = [
    MLFilterConfig(
        name="Baseline_Robot",
    )
]

if selected_filter_name != "Baseline_Robot":
    audit_filters.append(
        selected_filter_config
    )

audit_results, audit_outputs = run_filter_grid(
    probability_prices=audit_prices,
    filters=audit_filters,
    strategy_config=FINAL_STRATEGY_CONFIG,
    portfolio_config=FINAL_PORTFOLIO_CONFIG,
    start=AUDIT_START,
    end=AUDIT_END,
)

audit_results = add_baseline_differences(
    audit_results
)

display(
    audit_results[
        validation_columns
    ].sort_values(
        ["Calmar", "CAGR_%"],
        ascending=False,
    )
)


## 8. Audit döneminde BIST100 karşılaştırması


In [ ]:
audit_comparison_rows = []

for filter_name, output in audit_outputs.items():
    equity = output["equity"]
    trades = output["trades"]
    metrics = portfolio_metrics(
        equity,
        trades,
    )
    metrics["Portfolio"] = filter_name
    audit_comparison_rows.append(metrics)

baseline_audit_equity = audit_outputs[
    "Baseline_Robot"
]["equity"]

bist100_audit = build_benchmark_equity(
    market_prices=market_prices,
    comparison_dates=baseline_audit_equity["Date"],
    initial_capital=(
        FINAL_PORTFOLIO_CONFIG.initial_capital
    ),
    include_costs=False,
    benchmark_name="BIST100",
)

bist100_metrics = portfolio_metrics(
    bist100_audit,
    pd.DataFrame(columns=["Return"]),
)
bist100_metrics["Portfolio"] = "BIST100 Gross"
audit_comparison_rows.append(bist100_metrics)

audit_comparison = pd.DataFrame(
    audit_comparison_rows
)

display(
    audit_comparison[
        [
            "Portfolio",
            "End_Value",
            "Total_Return_%",
            "CAGR_%",
            "Max_Drawdown_%",
            "Sharpe",
            "Calmar",
            "Profit_Factor",
            "Trade_Count",
        ]
    ]
)


## 9. Sonuçları kaydet


In [ ]:
ML_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "ml"
)
ML_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

validation_results.to_csv(
    ML_RESULTS_DIR
    / "ml_portfolio_validation_grid.csv",
    index=False,
)

acceptance_table.to_csv(
    ML_RESULTS_DIR
    / "ml_portfolio_validation_acceptance.csv",
    index=False,
)

audit_results.to_csv(
    ML_RESULTS_DIR
    / "ml_portfolio_audit_result.csv",
    index=False,
)

audit_comparison.to_csv(
    ML_RESULTS_DIR
    / "ml_portfolio_audit_vs_bist100.csv",
    index=False,
)

decision = {
    "champion_model_name": champion_model_name,
    "selected_filter_name": selected_filter_name,
    "ml_accepted_on_validation": bool(ml_accepted),
    "validation_period": {
        "start": VALIDATION_START,
        "end": VALIDATION_END,
    },
    "audit_period": {
        "start": AUDIT_START,
        "end": AUDIT_END,
    },
    "decision_rule": (
        "ML is accepted only if it improves Validation CAGR and "
        "Calmar, does not reduce Profit Factor, retains at least "
        "50% of baseline trades and worsens drawdown by no more "
        "than 3 percentage points."
    ),
}

DECISION_PATH = (
    PROJECT_ROOT
    / "models"
    / "meta_label_portfolio_decision.json"
)

with DECISION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Karar kaydedildi:", DECISION_PATH)
print("Final karar:", decision)


## Nihai yorum

- `ml_accepted_on_validation=False` ise ML modeli araştırma sonucu olarak
  saklanır fakat günlük Robot sistemine bağlanmaz.
- `True` ise seçilen filtrenin Audit sonucu yalnızca ileri dönem kontrolüdür.
- Audit kötüleşirse ML canlı kullanıma alınmaz.
- ML kabul edilse bile gerçek yeni out-of-sample doğrulama paper trading ile
  yapılmalıdır.
